# 01 — Data Understanding
Load all 5 raw files, profile shapes/dtypes/missings/duplicates, check AttritionRisk class balance.

In [1]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW = r'../data/raw'

files = {
    'employee_attrition': 'employee_performance_pro.csv',
    'employee_engagement': 'Employee_Performance_Dataset.csv',
    'occupation': 'occupation_data.csv',
    'essential_skills': 'essential_skills.csv',
    'software_skills': 'software_skills.csv',
}

dfs = {}
for key, fname in files.items():
    df = pd.read_csv(f'{RAW}/{fname}')
    dfs[key] = df
    print(f"\n{'='*60}")
    print(f"FILE: {fname}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Dtypes:\n{df.dtypes.value_counts()}")
    missing = df.isnull().sum()
    if missing.any():
        print(f"Missing values:\n{missing[missing>0]}")
    else:
        print("Missing values: None")
    dups = df.duplicated().sum()
    print(f"Duplicate rows: {dups}")



FILE: employee_performance_pro.csv
Shape: (500, 24)
Columns: ['EmployeeID', 'Name', 'Gender', 'Age', 'Department', 'JobRole', 'EducationLevel', 'JoiningDate', 'CountryCode', 'Country', 'PhoneNumber', 'MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 'LastLeaveDate', 'LeaveDayName', 'ProjectsHandled', 'TrainingHours', 'CustomerSatisfaction', 'LastPromotionYear', 'YearsAtCompany', 'WorkLifeBalanceScore', 'PerformanceRating', 'AttritionRisk']
Dtypes:
int64      13
object      9
float64     2
Name: count, dtype: int64
Missing values:
CustomerSatisfaction    319
dtype: int64
Duplicate rows: 0

FILE: Employee_Performance_Dataset.csv
Shape: (5000, 13)
Columns: ['Employee ID', 'Name', 'Department', 'Job Role', 'Performance Score', 'KPI Score', 'Attendance (%)', 'Peer Rating', 'Task Completion (%)', 'Work Hours Logged', 'Manager Feedback', 'Training Hours', 'Promotion Eligibility']
Dtypes:
float64    5
int64      4
object     4
Name: count, dtype: int64
Missing values: None
Duplicate ro


FILE: software_skills.csv
Shape: (31821, 7)
Columns: ['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element ID', 'Element Name', 'Hot Technology', 'In Demand']
Dtypes:
object    7
Name: count, dtype: int64
Missing values: None
Duplicate rows: 0


In [2]:

# AttritionRisk class balance — critical for ML strategy
ea = dfs['employee_attrition']
print("\n=== AttritionRisk Class Balance ===")
vc = ea['AttritionRisk'].value_counts()
print(vc)
print(f"Minority class ratio: {vc.min()/len(ea):.2%}")
print("\nConclusion: If minority < 20%, prefer recall-optimized models + class_weight='balanced'")



=== AttritionRisk Class Balance ===
AttritionRisk
No     445
Yes     55
Name: count, dtype: int64
Minority class ratio: 11.00%

Conclusion: If minority < 20%, prefer recall-optimized models + class_weight='balanced'


In [3]:

# Basic stats for numeric columns
print("\n=== Employee Attrition — Numeric Summary ===")
print(ea.describe().round(2))
print("\n=== Engagement Dataset — Numeric Summary ===")
print(dfs['employee_engagement'].describe().round(2))



=== Employee Attrition — Numeric Summary ===
       EmployeeID     Age  EducationLevel  CountryCode   PhoneNumber  \
count      500.00  500.00          500.00       500.00  5.000000e+02   
mean       250.50   40.86            3.02        37.02  4.892320e+09   
std        144.48   12.09            1.42        31.55  2.876007e+09   
min          1.00   21.00            1.00         1.00  4.639131e+07   
25%        125.75   30.00            2.00         1.00  2.413195e+09   
50%        250.50   42.00            3.00        44.00  5.042778e+09   
75%        375.25   52.00            4.00        49.00  7.255667e+09   
max        500.00   60.00            5.00        91.00  9.961638e+09   

       MonthlySalary  OvertimeHoursPerMonth  LeavesTaken  ProjectsHandled  \
count         500.00                 500.00       500.00           500.00   
mean       103678.76                  20.04        13.72             8.07   
std         46043.11                  11.71         9.06             4.32 

In [4]:

# Identify potential ID columns
print("\n=== Potential ID columns (high cardinality) ===")
for key, df in dfs.items():
    id_cols = [c for c in df.columns if df[c].nunique() == len(df) or 'id' in c.lower() or 'ID' in c]
    print(f"{key}: {id_cols}")



=== Potential ID columns (high cardinality) ===
employee_attrition: ['EmployeeID', 'Name', 'PhoneNumber']
employee_engagement: ['Employee ID']
occupation: ['O*NET-SOC Code', 'Title', 'Description']
essential_skills: ['Element ID', 'Scale ID']
software_skills: ['Element ID']


In [5]:

# Unique value counts for categorical columns in attrition dataset
print("\n=== Categorical Unique Values (employee_attrition) ===")
cats = ea.select_dtypes(include='object').columns.tolist()
for c in cats:
    vals = ea[c].unique()[:10]
    print(f"  {c} ({ea[c].nunique()} unique): {vals}")



=== Categorical Unique Values (employee_attrition) ===
  Name (500 unique): ['Steven Barnett' 'Christopher Benson' 'Norman Lane' 'Rita Walker'
 'Judith Ware' 'Peggy Mann' 'Brenda Bauer' 'Troy Ford' 'Joel Fuller'
 'Cody Bright']
  Gender (3 unique): ['Other' 'Female' 'Male']
  Department (6 unique): ['Finance' 'Sales' 'Support' 'HR' 'IT' 'Marketing']
  JobRole (13 unique): ['Auditor' 'Sales Executive' 'Helpdesk' 'HR Executive' 'Account Manager'
 'Engineer' 'Accountant' 'HR Manager' 'Developer' 'SEO Analyst']
  JoiningDate (473 unique): ['2016-05-05' '2014-08-20' '2010-05-17' '2015-06-20' '2019-08-20'
 '2018-08-28' '2018-09-13' '2016-04-18' '2010-05-26' '2011-06-16']
  Country (6 unique): ['India' 'Canada' 'Germany' 'France' 'USA' 'UK']
  LastLeaveDate (268 unique): ['2024-07-03' '2024-01-05' '2024-11-27' '2024-07-15' '2024-12-17'
 '2024-10-01' '2024-06-01' '2024-10-26' '2024-07-20' '2024-07-26']
  LeaveDayName (7 unique): ['Wednesday' 'Friday' 'Monday' 'Tuesday' 'Saturday' 'Thursday' '

**Summary**: Loaded 5 files. AttritionRisk is the ML target. Class balance checked. ID columns identified. Categorical columns listed for downstream encoding.